# Data Preprocessing: Correlation Analysis
**Mục tiêu**: Kiểm tra ma trận tương quan giữa các biến đầu vào và biến mục tiêu `NDVI_Season_Mean` để chuẩn bị cho quá trình huấn luyện mô hình.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Tải dữ liệu
df = pd.read_csv('../Data Collection/Bangladesh_database_Final_Merged.csv')
df.head()

## 1. Tính toán ma trận tương quan
Chúng ta sẽ lọc ra các biến dạng số (numeric) để tính ma trận tương quan Pearson.

In [ ]:
# Lọc các cột dạng số (numeric)
numeric_df = df.select_dtypes(include=[np.number])

# Tính ma trận tương quan Pearson
corr_matrix = numeric_df.corr()

# Lấy ra mức độ tương quan của các biến với biến mục tiêu NDVI_Season_Mean
corr_with_target = corr_matrix['NDVI_Season_Mean'].sort_values(ascending=False)

print("Tương quan với NDVI_Season_Mean:")
print(corr_with_target)

## 2. Trực quan hóa bằng Heatmap
Heatmap giúp chúng ta nhìn nhận rõ ràng hơn sự tương quan giữa TẤT CẢ các cặp biến.

In [ ]:
plt.figure(figsize=(20, 16))
# Vẽ heatmap cho ma trận tương quan
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0, linewidths=0.5)
plt.title('Ma trận tương quan giữa các biến số (Correlation Heatmap)', fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
# Trực quan hóa riêng với biến mục tiêu
sns.barplot(x=corr_with_target.values, y=corr_with_target.index, palette='viridis')
plt.title('Mức độ tương quan của các biến đầu vào với NDVI_Season_Mean', fontsize=16)
plt.xlabel('Hệ số tương quan (Pearson)')
plt.ylabel('Biến số')
plt.axvline(x=0, color='red', linestyle='--')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

## 3. Đánh giá sơ bộ và Định hướng Preprocessing

Dựa vào biểu đồ và các hệ số tương quan với `NDVI_Season_Mean`, chúng ta có thể đưa ra các lập luận sau:

1.  **Các biến tương quan thuận (Positive Correlation)**:
    *   **LAI (Leaf Area Index), FPAR, EVI**: Thường mang tương quan dương cực mạnh với NDVI vì chúng cùng phản ánh sức khỏe của thảm thực vật. Tuy nhiên, việc giữ lại toàn bộ các chỉ số viễn thám này sẽ dẫn đến vấn đề **Đa cộng tuyến (Multicollinearity)**. Thông thường chỉ nên chọn 1-2 chỉ số đại diện.
    *   **Rainfall / Soil_Moisture_mm**: Lượng mưa và độ ẩm đất cung cấp nước cho cây phát triển, tương quan thuận với NDVI là hợp lý.

2.  **Các biến tương quan nghịch (Negative Correlation)**:
    *   **LST_Kelvin (Nhiệt độ bề mặt)**: Đất càng cằn cỗi và thảm thực vật càng thưa thớt (NDVI thấp) thì bề mặt càng hấp thụ bức xạ nhiệt mạnh (LST cao). Do đó tương quan nghịch là chính xác 100% về mặt vật lý viễn thám.
    *   **Avg_Salinity_Index**: Độ mặn kìm hãm sự phát triển của cây trồng, mang hệ số tương quan nghịch là hoàn toàn hợp lý.
    *   **Temp_Max / Heat_Stress_Days**: Nhiệt độ tối đa quá cao hoặc kéo dài làm cây bị sốc nhiệt, sinh trưởng kém.

3.  **Vấn đề Đa cộng tuyến (Multicollinearity)** cần xử lý:
    *   Nhiệt độ (`Temp_Mean`, `Temp_Max`, `Temp_Min`) tương quan rất mạnh với nhau.
    *   Các chỉ số (`EVI`, `LAI`, `FPAR`, `NDVI_Season_Mean`) cũng tương quan mạnh chéo nhau.
    *   **Giải pháp cho bước tiếp theo trong Preprocessing**: Chúng ta có thể sẽ cần Drop một vài biến dư thừa (như Temp_Mean hoặc giữ lại EVI/LAI và drop bớt cái còn lại) hoặc áp dụng PCA / Feature Selection để giảm thiểu hiện tượng Overfitting.